# 吉祥寺で3つのPOIデータを比較する

## 1. 今回やること
Foursquare / Overture / OpenStreetMapを同じ吉祥寺AOIで並べ、基本属性と同じPOIらしい候補を確認します。件数だけで優劣を決めません。

[利用手順](../docs/poi-comparison.md) / [検証記録](../docs/poi-comparison-validation.md) を先に確認してください。

## 2. 3データを準備
Python / uvとNotebook・OSM依存グループを準備してください。Overture・OSMは各編で保存した結果を再利用します。Foursquare全件結果がない場合だけ、プロジェクトの`.env`の`FSQ_OS_PLACES_TOKEN`で指定snapshotから取得します。

COUNT・取得行数・unique ID・保存件数が一致しない場合は停止します。前回の500件サンプルは使いません。

In [ ]:
import pandas as pd
from geoai_open_lab.poi_comparison import (
    prepare_inputs, attribute_presence, run_comparison,
)

inputs = prepare_inputs(acquire_foursquare=False)

In [ ]:
versions = []
for source, p in inputs["provenance"].items():
    datasets = p["snapshots"] or {"places": {"snapshot_id": p["release"] or p["base_timestamp"]}}
    for dataset, details in datasets.items():
        versions.append({"provider": source, "dataset": dataset,
                         "version": details["snapshot_id"],
                         "acquisition UTC": p["acquisition_start_utc"]})
pd.DataFrame(versions)

## 3. 比較用ビュー
元名称・座標・住所・電話・Webサイト・元カテゴリを比較用の小さな表へ変換済みです。カテゴリ共通化はしません。OSM住所は採用済みIDに限り、保存済み応答から補完しています。

In [ ]:
views = inputs["views"]
pd.DataFrame({"provider": list(views), "records": [len(v) for v in views.values()]})

## 4. 基本件数と属性presence
分母は各データの取得全レコードです。presenceは値の存在だけを表し、正確性・最新性ではありません。住所は市名等の一部だけでも値ありです。FoursquareのbrandはN/Aです。

In [ ]:
presence = attribute_presence(views)
presence.pivot(index="source", columns="attribute", values="percent").round(1).fillna("N/A")

## 5. POIの一致候補
100m以内の全候補を作り、非空の正規化名完全一致・30m以内・双方で一意・競合や明確な属性矛盾なし、を高信頼候補とします。類似名や競合は要確認です。補助属性だけでは昇格しません。

以下はローカル計算です。候補表・距離分布・確認用CSV・manifestはGit管理外へ保存します。少し時間がかかります。

In [ ]:
result = run_comparison(inputs)
summary = result["summary"]

## 6. 重なり
候補ペア数、採用した高信頼候補、要確認候補を分けます。three-wayは3辺すべてが高信頼の組だけです。

In [ ]:
pd.DataFrame([
    {"pair": pair, "candidate": values["candidate_pairs"],
     "high confidence": values["high_confidence"], "ambiguous": values["ambiguous_pairs"]}
    for pair, values in summary["pairs"].items()
])

In [ ]:
print("three-way候補:", summary["three_way"])
print("手動確認:", "pending（未確認）")
print("保存先:", result["path"].relative_to(result["path"].parents[3]).as_posix())

## 7. 注意点
- snapshotは完全同時刻ではなく、POIの定義・更新時点・geometryの扱いも異なります。
- 営業状態の共通filterはなく、NULLを営業中とは扱いません。
- OvertureにはFoursquare由来sourceがあり、両者の一致は独立した裏付けではありません。
- unmatchedは他データに存在しないことを意味しません。
- 高信頼はルールによる候補です。人間の確認用sampleはpendingです。
- 正解データがないためaccuracy / coverageや優劣は評価しません。
- データと対応表はGitHubへ公開せず、外部データの元ライセンスを維持します。

© OpenStreetMap contributors。Foursquare・Overtureを含む利用条件は[ガイド](../docs/poi-comparison.md)を参照してください。